In [3]:
# Install dependencies
!pip install gtts elevenlabs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 9.7 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [8]:
from gtts import gTTS
from elevenlabs.client import ElevenLabs
from elevenlabs import save
from IPython.display import Audio, display

def speak(text, use_elevenlabs=True, api_key=None):
    if use_elevenlabs and api_key:
        try:
            client = ElevenLabs(api_key=sk_829c0f6dec8ad207a4d2f9d4607395e877e4a4d1bbcee640)
            audio = client.text_to_speech.convert(
                text=text,
                voice_id="vghiSqG5ezdhd8F3tKAD",
                model_id="eleven_multilingual_v2",
                output_format="mp3_44100_128",
            )
            save(audio, "output.mp3")
            print("✅ ElevenLabs audio saved!")
            display(Audio("output.mp3", autoplay=True))

        except Exception as e:
            print(f"⚠️ ElevenLabs failed: {e}. Falling back to gTTS...")
            _gtts_speak(text)
    else:
        _gtts_speak(text)

def _gtts_speak(text):
    tts = gTTS(text=text, lang='en')
    tts.save("output_gtts.mp3")
    print("✅ gTTS audio saved!")
    display(Audio("output_gtts.mp3", autoplay=True))


speak("Janab! Apka paisa pending hai usse jaldi bhare", use_elevenlabs=True, api_key="sk_829c0f6dec8ad207a4d2f9d4607395e877e4a4d1bbcee640")

⚠️ ElevenLabs failed: name 'sk_829c0f6dec8ad207a4d2f9d4607395e877e4a4d1bbcee640' is not defined. Falling back to gTTS...
✅ gTTS audio saved!


In [9]:
!pip install fastapi uvicorn elevenlabs python-multipart

In [10]:
from fastapi import FastAPI, Query
from fastapi.responses import FileResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from elevenlabs.client import ElevenLabs
from elevenlabs import save
import os

app = FastAPI()

# ── Allow web devs to call from any domain (CORS) ──────────────────────
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # or restrict to ["https://yourwebsite.com"]
    allow_methods=["*"],
    allow_headers=["*"],
)

API_KEY = "sk_829c0f6dec8ad207a4d2f9d4607395e877e4a4d1bbcee640"
client = ElevenLabs(api_key=API_KEY)

@app.get("/tts")
def text_to_speech(text: str = Query(..., description="Text to convert to speech")):
    try:
        audio = client.text_to_speech.convert(
            text=text,
            voice_id="vghiSqG5ezdhd8F3tKAD",

            model_id="eleven_multilingual_v2",
            output_format="mp3_44100_128",
        )
        output_path = "output.mp3"
        save(audio, output_path)

        return FileResponse(
            output_path,
            media_type="audio/mpeg",
            filename="speech.mp3"
        )
    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})

@app.get("/")
def root():
    return {"message": "TTS API is running!"}